# **인공신경망과 딥러닝**


---



**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
# 코랩에서 실행
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shutil
import warnings
import logging

# ── 1. 나눔 폰트 설치 ──────────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1

# ── 2. matplotlib 캐시 디렉터리 삭제 ──────────────────────────
shutil.rmtree(mpl.get_cachedir(), ignore_errors=True)

# ── 3. 폰트 등록 및 유니코드 마이너스 경고 방지 ──────────────────
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (6, 4)

# 수학 기호(마이너스 등)를 일반 텍스트로 처리하도록 설정하여 폰트 에러 차단
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.default'] = 'regular'

# mathtext 및 font_manager 관련 폰트 결함 경고 완전히 무시 설정
logging.getLogger('matplotlib.mathtext').setLevel(logging.ERROR)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib.font_manager')



---



# **2. PyTorch로 딥러닝하기** + 깊은 신경망의 문제와 해결

- 학습목표
    - Tensor·autograd·`nn.Module`·`DataLoader`로 **PyTorch 표준 학습 루프**를 작성한다
    - PyTorch 내부 동작을 이해한다.
    - 기울기 소실을 직접 관찰하고 ReLU·He 초기화·BatchNorm·잔차연결·Adam(W)의 효과를 비교한다.
- 사용 데이터셋
    - **Fashion-MNIST** (Zalando 실제 상품 이미지, Xiao et al., 2017)
- GPU 사용


In [ ]:
# [0] 공통 설정
import os, time, numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

DATA_DIR = os.environ.get("DATA_DIR", "./data")          # Colab: ./data 에 자동 다운로드
device = "cuda" if torch.cuda.is_available() else "cpu"   # GPU가 있으면 자동 사용
torch.manual_seed(42)
print("torch", torch.__version__, "| device:", device)

---
## **2-1. Tensor**: (NumPy 배열 + GPU + 자동미분)
| 개념 | NumPy | PyTorch |
|---|---|---|
| 생성 | `np.array([[1,2]])` | `torch.tensor([[1,2]])` |
| 모양 | `.shape` | `.shape` / `.size()` |
| 행렬곱 | `A @ B` | `A @ B` |
| 장치 이동 | – | `.to("cuda")` |
| 자동미분 | – | `requires_grad=True` → `.backward()` |

In [ ]:
# [Lab 2-1]  Tensor 체험
x = torch.arange(6, dtype=torch.float32).reshape(2, 3)    # 2x3 행렬
W = torch.randn(3, 4, requires_grad=True)                 # 학습할 가중치
y = (x @ W).relu().sum()                                  # 순전파
y.backward()                                              # 역전파 → W.grad 자동 계산
print("x.shape", x.shape, "| (x@W).shape", (x @ W).shape, "| W.grad.shape", W.grad.shape)
print("NumPy 변환:", x.numpy()[0], "| 브로드캐스팅:", (x + torch.tensor([10., 20., 30.]))[0])

---
## **2-2. 데이터 준비**: Dataset → DataLoader (미니배치)

In [ ]:
# [Lab 2-2] Fashion-MNIST 로딩 (Colab에서 최초 1회 다운로드, 약 30MB)
# 1.데이터 전처리
tf = transforms.Compose([transforms.ToTensor(),                       # [0,255] → [0,1], (1,28,28)
                         transforms.Normalize((0.2860,), (0.3530,))])  # 학습셋 평균·표준편차로 표준화

# 2.데이터셋 로드
train_full = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=tf)
test_ds    = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=tf)

# 학습 54,000 / 검증 6,000 분리 (테스트셋은 마지막 평가에만 사용!)
# 3.학습 및 검증 데이터 분리
g = torch.Generator().manual_seed(0)
train_ds, val_ds = torch.utils.data.random_split(train_full, [54000, 6000], generator=g)

# 4.미니배치 데이터 로더 구축
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=512)
test_dl  = DataLoader(test_ds, batch_size=512)

# 5.데이터 시각화
CLASSES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]
xb, yb = next(iter(train_dl)) # 첫 번째 미니배치(이미지 128장, 레이블 128개)를 가져와 크기를 출력
print("배치 모양:", xb.shape, yb.shape)
fig, axes = plt.subplots(1, 8, figsize=(12, 2)) # 가져온 미니배치 중 앞의 8개 이미지를 matplotlib에 시각화
for ax, img, lab in zip(axes, xb, yb):
    ax.imshow(img[0], cmap="gray")
    ax.set_title(CLASSES[lab], fontsize=9)
    ax.axis("off")
plt.show()

---
## **2-3. 모델 정의**: `nn.Module`

In [ ]:
# [Lab 2-3] MLP 모델: 784 → 256 → 10
class MLP(nn.Module):
    def __init__(self, n_in=784, n_hidden=256, n_out=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                 # (B,1,28,28) → (B,784)
            nn.Linear(n_in, n_hidden), nn.ReLU(),
            nn.Linear(n_hidden, n_out),   # 출력은 logits (softmax는 손실함수 안에 포함)
        )
    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
print(model)
print("학습 파라미터 수:", sum(p.numel() for p in model.parameters()))   # 784*256+256 + 256*10+10

---
## **2-4. PyTorch 표준 학습 루프** — **딥러닝의 5줄**
```python
optimizer.zero_grad()      # ① 이전 기울기 초기화
logits = model(x)          # ② 순전파
loss = loss_fn(logits, y)  # ③ 손실 계산
loss.backward()            # ④ 역전파 (autograd)
optimizer.step()           # ⑤ 가중치 갱신
```

In [ ]:
# [Lab 2-4] 학습/평가 함수 — 이후 CNN·RNN 실습에서도 그대로 재사용
def train_one_epoch(model, dl, loss_fn, opt):
    model.train()                                   # Dropout/BN을 학습 모드로
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        opt.step()
        loss_sum += loss.item() * len(y); total += len(y)
        correct += (logits.argmax(1) == y).sum().item()
    return loss_sum / total, correct / total

@torch.no_grad()                                    # 평가 때는 기울기 계산 불필요
def evaluate(model, dl, loss_fn):
    model.eval()                                    # Dropout 끄기, BN은 이동평균 사용
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss_sum += loss_fn(logits, y).item() * len(y); total += len(y)
        correct += (logits.argmax(1) == y).sum().item()
    return loss_sum / total, correct / total

loss_fn = nn.CrossEntropyLoss()                     # = LogSoftmax + NLLLoss (정수 레이블 그대로 사용)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
hist = []
for ep in range(1, 4):                              # CPU에서 에폭당 약 10~20초
    t0 = time.time()
    tr = train_one_epoch(model, train_dl, loss_fn, opt)
    va = evaluate(model, val_dl, loss_fn)
    hist.append((*tr, *va))
    print(f"epoch {ep} | train loss {tr[0]:.4f} acc {tr[1]:.4f} | val loss {va[0]:.4f} acc {va[1]:.4f} | {time.time()-t0:.1f}s")

### [참고] Keras ↔ PyTorch 대응표
| Keras  | PyTorch  | 비고 |
|---|---|---|
| `Sequential([Dense(100, activation='relu')])` | `nn.Sequential(nn.Linear(784,100), nn.ReLU())` | 입력 크기를 직접 명시 |
| `Dense(10, activation='softmax')` + `sparse_categorical_crossentropy` | `nn.Linear(100,10)` + `nn.CrossEntropyLoss()` | softmax가 손실 안에 포함 |
| `model.compile(optimizer='adam')` | `torch.optim.Adam(model.parameters())` | |
| `model.fit(x, y, epochs=5)` | 위의 **5줄 학습 루프**를 직접 작성 | 내부가 보인다! |
| `model.evaluate()` | `model.eval()` + `torch.no_grad()` | |
| `EarlyStopping` | 검증 손실을 보고 직접 `break` + `state_dict` 저장 | Lab 8에서 구현 |

In [ ]:
# [Lab 2-4-2] 테스트 평가 · 저장 · 불러오기 · 예측
test_loss, test_acc = evaluate(model, test_dl, loss_fn)
print(f"테스트 정확도: {test_acc:.4f}")

torch.save(model.state_dict(), "mlp_fashion.pt")       # 가중치만 저장 (권장 방식)
model2 = MLP().to(device); model2.load_state_dict(torch.load("mlp_fashion.pt", map_location=device))
model2.eval()
x, y = next(iter(test_dl))
with torch.no_grad():
    prob = F.softmax(model2(x[:5].to(device)), dim=1)
for p_, t in zip(prob, y[:5]):
    print(f"정답 {CLASSES[t]:<9} → 예측 {CLASSES[p_.argmax()]:<9} (확률 {p_.max():.2f})")



---


### **[실습] 은닉 뉴런 갯수에 따른 검증 정확도 비교**
위 코드에서 은닉 뉴런을 64 / 512로 바꾸고 파라미터 수와 **검증 정확도**를 표로 비교하라

|은닉 뉴런|파라미터 수|검증 정확도|
|---|---|---|
|n_hidden=256 | 203530| 0.8820|
|n_hidden=64 | | |
|n_hidden=512 |  | |


In [ ]:
# [Lab 2-3] MLP 모델: 784 → ? → 10







---


### **[실습] MLP에 Dropout layer 추가하고 정확도 비교**
MLP에 `nn.Dropout(0.3)`을 추가하고 10 epoch 학습해 학습/검증 정확도 차이(과대적합)를 Dropout 유무로 비교하라

In [ ]:
# 앞에서 작성한 MLP에 nn.Dropout(0.3)을 추가하고 10 epoch 학습해 학습/검증 정확도 차이(과대적합)를 Dropout 유무로 비교하라







print("\n=== 최종 결과 비교 ===")
print(f"[Dropout 미사용] 학습 정확도: {tr_acc_no:.4f} | 검증 정확도: {va_acc_no:.4f} | 차이(격차): {tr_acc_no - va_acc_no:+.4f}")
print(f"[Dropout 0.3 사용] 학습 정확도: {tr_acc_yes:.4f} | 검증 정확도: {va_acc_yes:.4f} | 차이(격차): {tr_acc_yes - va_acc_yes:+.4f}")

---
## **2-5. 활성화 함수의 진화**: Sigmoid → ReLU → GELU/SiLU (LLM에서 사용)
| 함수 | 등장 | 특징 | 대표 사용처 |
|---|---|---|---|
| Sigmoid | 1980s | 출력 (0,1), 최대 기울기 0.25 → **기울기 소실** | 이진 분류 출력층, 게이트 |
| Tanh | 1980s | 출력 (-1,1), 0 중심 | RNN 은닉층 |
| ReLU | 2010 (Nair & Hinton) | 양수 구간 기울기 1, 계산 저렴 | CNN·MLP 기본값 |
| GELU | 2016 (Hendrycks & Gimpel) | 부드러운 ReLU | BERT·GPT 계열 |
| SiLU/Swish | 2017 (Ramachandran et al.) | $x\cdot\sigma(x)$ | LLaMA·Qwen의 SwiGLU |

In [ ]:
# [Lab 2-5] 활성화 함수와 도함수 비교
z = torch.linspace(-5, 5, 400, requires_grad=True)
acts = {"Sigmoid": torch.sigmoid,
        "Tanh": torch.tanh,
        "ReLU": F.relu,
        "GELU": F.gelu,
        "SiLU": F.silu}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, f in acts.items():
    out = f(z); grad, = torch.autograd.grad(out.sum(), z)
    axes[0].plot(z.detach(), out.detach(), label=name); axes[1].plot(z.detach(), grad, label=name)
axes[0].set_title("activation f(z)")
axes[1].set_title("derivative f'(z)  (sigmoid max = 0.25)")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.show()

---
## **2-6. 깊게 쌓으면 생기는 문제** — 역사 재현 실험실 (Lab 4)
| 연도 | 막힘(Problem) | 돌파(Breakthrough) |
|---|---|---|
| 1991 | 층이 깊어질수록 앞쪽 층의 기울기가 0에 수렴 (Hochreiter) | – |
| 2010 | 초기화가 잘못되면 신호가 사라지거나 폭발 | Xavier 초기화 (Glorot & Bengio), ReLU (Nair & Hinton) |
| 2012~14 | 과대적합 | Dropout (Hinton et al. 2012; Srivastava et al. 2014) |
| 2015 | 층마다 입력 분포가 흔들림 | BatchNorm (Ioffe & Szegedy), He 초기화 (He et al.) |
| 2015 | 아주 깊으면 오히려 학습이 안 됨 | 잔차연결 ResNet (He et al.) |



### 실험1: 10층 MLP 생성
같은 10층 MLP에 처방을 하나씩 더해 **층별 기울기 크기**와 **학습 곡선**을 비교합니다.
(시간 절약을 위해 학습 데이터 10,000장, 2에폭, SGD로 실험)

In [ ]:
# [Lab 2-6-1] 실험용 깊은 MLP 생성기
class DeepMLP(nn.Module):
    def __init__(self, act="sigmoid", init="small", bn=False, residual=False, depth=10, width=128):
        super().__init__()
        self.inp = nn.Linear(784, width)
        self.hidden = nn.ModuleList([nn.Linear(width, width) for _ in range(depth)])
        self.bns = nn.ModuleList([nn.BatchNorm1d(width) if bn else nn.Identity() for _ in range(depth)])
        self.out = nn.Linear(width, 10)
        self.act = {"sigmoid": torch.sigmoid, "relu": F.relu}[act]
        self.residual = residual
        for layer in [self.inp, *self.hidden]:
            if init == "small":                                  # 1990년대식: 작은 정규분포
                nn.init.normal_(layer.weight, 0, 0.05)
            elif init == "he":                                   # He(2015): ReLU에 맞춘 분산 2/fan_in
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        h = self.act(self.inp(x.flatten(1)))
        for lin, bn in zip(self.hidden, self.bns):
            if self.residual:
                h = h + lin(self.act(bn(h)))     # 잔차연결(pre-activation): 입력 h를 그대로 더해 '지름길' 확보
            else:
                h = self.act(bn(lin(h)))
        return self.out(self.act(h) if self.residual else h)

CONFIGS = {
    "A. Sigmoid + small init (1990s)": dict(act="sigmoid", init="small"),
    "B. ReLU + He init (2010-15)":     dict(act="relu", init="he"),
    "C. B + BatchNorm (2015)":         dict(act="relu", init="he", bn=True),
}
small_train = Subset(train_ds, range(10000))
small_dl = DataLoader(small_train, batch_size=128, shuffle=True)

In [ ]:
# [Lab 2-6-2] 첫 배치에서 층별 기울기 크기 측정 + 2에폭 학습 곡선
def layer_grad_norms(model):
    x, y = next(iter(DataLoader(small_train, batch_size=256)))
    model.zero_grad()
    F.cross_entropy(model(x.to(device)), y.to(device)).backward()
    return [l.weight.grad.norm().item() for l in [model.inp, *model.hidden]]

results = {}
for name, cfg in CONFIGS.items():
    torch.manual_seed(0)
    m = DeepMLP(**cfg).to(device)
    gnorm = layer_grad_norms(m)
    opt = torch.optim.SGD(m.parameters(), lr=0.05)
    curve = []
    for ep in range(2):
        m.train()
        for i, (x, y) in enumerate(small_dl):
            opt.zero_grad()
            loss = F.cross_entropy(m(x.to(device)), y.to(device))
            loss.backward()
            opt.step()
            curve.append(loss.item())
    acc = evaluate(m, val_dl, loss_fn)[1]
    results[name] = (gnorm, curve, acc)
    print(f"{name:<34} 1층 기울기 {gnorm[0]:.2e} | 마지막층 {gnorm[-1]:.2e} | 검증 정확도 {acc:.3f}")

In [ ]:
# 시각화: (좌) 층별 기울기 크기(로그) (우) 학습 손실 곡선
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, (g_, c_, a_) in results.items():
    axes[0].plot(range(1, len(g_) + 1), g_, marker="o", label=name)
    axes[1].plot(np.convolve(c_, np.ones(10) / 10, mode="valid"), label=f"{name} (val {a_:.2f})")
axes[0].set_yscale("log")
axes[0].set_xlabel("layer (input → output)")
axes[0].set_ylabel("||dL/dW||")
axes[0].set_title("Gradient norm per layer")
axes[1].set_xlabel("iteration")
axes[1].set_ylabel("train loss")
axes[1].set_title("Training curve (SGD, lr=0.05)")

for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 실험2: 30층 MLP 생성 → 아주 깊으면 오히려 학습이 안 됨



In [ ]:
# [Lab 2-6-3] 더 깊게(30층): BatchNorm만 vs BatchNorm + 잔차연결(ResNet 아이디어)
for name, cfg in {"C. ReLU+He+BN (30 layers)": dict(act="relu", init="he", bn=True, depth=30),
                  "D. C + Residual (30 layers)": dict(act="relu", init="he", bn=True, residual=True, depth=30)}.items():
    torch.manual_seed(0)
    m = DeepMLP(**cfg).to(device)
    opt = torch.optim.SGD(m.parameters(), lr=0.05)
    for ep in range(2):
        m.train()
        for x, y in small_dl:
            opt.zero_grad()
            F.cross_entropy(m(x.to(device)), y.to(device)).backward()
            opt.step()
    print(f"{name:<30} 검증 정확도 {evaluate(m, val_dl, loss_fn)[1]:.3f}")

**관찰 포인트**
| 질문 | 확인할 것 |
|---|---|
| A의 1층 기울기는 마지막 층보다 몇 자릿수 작은가? | 시그모이드 미분 최대 0.25가 10번 곱해지는 효과 |
| B~D는 왜 학습 곡선이 빨리 떨어지는가? | 기울기가 층을 지나도 크기가 유지됨 |
| 30층에서 잔차연결(D)이 BN만 쓴 C보다 나은 이유는? LLM에서도 쓰이는 이유는? | Transformer 블록도 `x + Attention(x)`, `x + FFN(x)` 구조 |

---
## **2-7. 옵티마이저의 진화**: SGD → Momentum → Adam → AdamW
| 옵티마이저 | 아이디어 | 출처 |
|---|---|---|
| SGD | 기울기 반대 방향으로 이동 | Robbins & Monro (1951) |
| Momentum | 이전 이동 방향을 관성으로 유지 | Polyak (1964) |
| Adam | 1차·2차 모멘트로 파라미터별 학습률 조정 | Kingma & Ba (2015) |
| AdamW | 가중치 감쇠를 기울기와 분리 → LLM 학습 표준 | Loshchilov & Hutter (2019) |
| Muon 등 (최신) | 행렬 단위 직교화 업데이트, 대규모 LLM 학습에 적용 사례 | Kimi K2 기술보고서 (2025) |

In [ ]:
# [Lab 2-7] 같은 모델(B 구성)에 옵티마이저만 바꿔 비교
OPTS = {
    "SGD":            lambda p: torch.optim.SGD(p, lr=0.05),
    "SGD+Momentum":   lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
    "Adam":           lambda p: torch.optim.Adam(p, lr=1e-3),
    "AdamW":          lambda p: torch.optim.AdamW(p, lr=1e-3, weight_decay=1e-2),
}
plt.figure(figsize=(8, 4))
for name, make in OPTS.items():
    torch.manual_seed(0)
    m = DeepMLP(act="relu", init="he", depth=4).to(device)
    opt = make(m.parameters())
    curve = []
    for ep in range(2):
        m.train()
        for x, y in small_dl:
            opt.zero_grad()
            loss = F.cross_entropy(m(x.to(device)), y.to(device))
            loss.backward()
            opt.step()
            curve.append(loss.item())
    acc = evaluate(m, val_dl, loss_fn)[1]
    plt.plot(np.convolve(curve, np.ones(10) / 10, mode="valid"), label=f"{name} (val {acc:.3f})")
plt.yscale("log")
plt.xlabel("iteration")
plt.ylabel("train loss")
plt.title("Optimizer comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
### 참고문헌
- Xiao, H., Rasul, K., & Vollgraf, R. (2017). Fashion-MNIST. arXiv:1708.07747.
- Hochreiter, S. (1991). *Untersuchungen zu dynamischen neuronalen Netzen* (Diploma thesis, TU München).
- Glorot, X., & Bengio, Y. (2010). Understanding the difficulty of training deep feedforward neural networks. *AISTATS*.
- Nair, V., & Hinton, G. E. (2010). Rectified linear units improve restricted Boltzmann machines. *ICML*.
- Srivastava, N., et al. (2014). Dropout. *JMLR*, 15, 1929–1958.
- Ioffe, S., & Szegedy, C. (2015). Batch normalization. *ICML*.
- He, K., et al. (2015). Delving deep into rectifiers. *ICCV*. / He, K., et al. (2016). Deep residual learning. *CVPR*.
- Kingma, D. P., & Ba, J. (2015). Adam. *ICLR*. / Loshchilov, I., & Hutter, F. (2019). Decoupled weight decay regularization. *ICLR*.
- Hendrycks, D., & Gimpel, K. (2016). GELU. arXiv:1606.08415.
- Kimi Team (2025). Kimi K2: Open Agentic Intelligence. arXiv:2507.20534.
- PyTorch 공식 문서: https://docs.pytorch.org/docs/stable/